# 05a2 deep pilot fix

Phase 5a2 for the battery SOH nowcast: position-aware CNN (M1p), inner-fold pilot on 3 outer sets x 2 seeds, exact Phase 4 comparators, primary deep model choice for Phase 5b.
Scores no outer-test cell, touches no batch-3 or sensitivity cell, writes no gate file. Writes only to `outputs/phase5a2` and `src/soh_deep_v2.py`.

## 1. Setup and guards

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must precede torch import
import sys, json, time, math, gc, hashlib, platform, warnings, traceback, importlib, ctypes, statistics
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, psutil
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

T0 = time.time()
IS_WIN = platform.system() == "Windows"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data/processed"; SPL = PROC / "splits"; P3 = ROOT / "outputs/phase3"
OUT4 = ROOT / "outputs/phase4"; OUT5A = ROOT / "outputs/phase5a"      # read-only
OUT = ROOT / "outputs/phase5a2"; FIG = OUT / "figures"; CACHE = OUT / "cache"; SRC = ROOT / "src"
for d in (OUT, FIG, CACHE):
    d.mkdir(parents=True, exist_ok=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

GATE_SHA = "40df2fb82096e994f8bf7b8084433bb5ca148d3fa92c780461f68c0c4247d901"
STITCHED = {"b1c0", "b1c1", "b1c2", "b1c3", "b1c4"}
SENSITIVITY = {"b1c8", "b1c10", "b1c12", "b1c13", "b1c22"}
WINDOW_V = {"W300": (3.5, 3.0), "W240": (3.5, 2.4)}
WINDOW_IDX = {"W300": (0, 333), "W240": (0, 733)}
KINDS = ("M0", "M1", "M1p")
VARIANTS = [("M0", "abs"), ("M0", "res"), ("M1", "abs"), ("M1p", "abs")]

BASE = dict(SEED=42, NOMINAL_AH=1.1, TRAIN_STRIDE=5, BATCH=256, CLIP=1.0, HUBER_DELTA=1.0,
            WD=1e-4, NOISE_SIGMA=0.02, WARMUP_FRAC=0.05)
FULL_CFG = dict(BASE, name="full", E=120, LRS=[2e-3, 4e-3], SEEDS=[42, 43], WINDOWS=["W300", "W240"],
                OUTER_SETS=[(0, 0), (0, 1), (0, 2)], TREES=150, LOG_EVERY=10,
                out=OUT, fig=FIG, cache=CACHE)
SMOKE_CFG = dict(BASE, name="smoke", E=2, LRS=[2e-3], SEEDS=[42], WINDOWS=["W300"],
                 OUTER_SETS=[(0, 0)], TREES=20, LOG_EVERY=1,
                 out=OUT / "smoke", fig=OUT / "smoke/figures", cache=OUT / "smoke/cache")

# ---- check / error / run log helpers
SCOPE = "setup"
CHECKS, ERRORS, RUNS = [], [], []

def check(name, passed, detail=""):
    """passed: True -> PASS, False -> FAIL, or the string 'WARN'."""
    status = passed if isinstance(passed, str) else ("PASS" if passed else "FAIL")
    CHECKS.append(dict(scope=SCOPE, name=name, status=status, detail=str(detail)))
    print(f"[{status}] {name} {detail}")
    return status == "PASS"

def record_error(stage, exc):
    tb = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    ERRORS.append(dict(scope=SCOPE, stage=stage, error=f"{type(exc).__name__}: {exc}", trace=tb))
    print(f"[ERROR] {stage}: {type(exc).__name__}: {exc}")

def log_run(group, name, seconds, cached):
    """Append to run_log.txt; WARN if > 3x the median of earlier real runs in the same group
    (grouped by model family so fast MLP runs do not make CNN runs look slow)."""
    prev = [r["seconds"] for r in RUNS if r["group"] == group and not r["cached"] and r["scope"] == SCOPE]
    if not cached and len(prev) >= 1 and seconds > 3 * statistics.median(prev):
        check(f"slow run {name}", "WARN", f"{seconds:.1f}s > 3x median {statistics.median(prev):.1f}s")
    RUNS.append(dict(scope=SCOPE, group=group, name=name, seconds=float(seconds), cached=bool(cached)))
    with open(OUT / "run_log.txt", "a") as f:
        f.write(f"{datetime.now().isoformat(timespec='seconds')}\t{SCOPE}\t{name}\t{seconds:.1f}s\t{'cached' if cached else 'run'}\n")

def keep_awake(on):
    if IS_WIN:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if on else 0x80000000)
keep_awake(True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def locate(name, dirs):
    for d in dirs:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in any of: {[str(d) for d in dirs]}")

In [2]:
# ---- gate and split hash checks
GATE_PATH = locate("week2_gate.json", [OUT4, P3, PROC])
GATE_HASH_PATH = locate("gate_hash.txt", [GATE_PATH.parent, OUT4, P3, PROC])

def gate_state():
    cur = sha256_file(GATE_PATH)
    txt = GATE_HASH_PATH.read_text().strip().split()[0].lower()
    return cur, txt

cur, txt = gate_state()
check("gate sha256 == gate_hash.txt (start)", cur == txt, cur[:16])
check("gate sha256 == frozen Phase 4 hash (start)", cur == GATE_SHA)
GATE_START = cur

def find_hash(obj, fname):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace("\\", "/").endswith(fname):
                return v if isinstance(v, str) else (v.get("sha256") if isinstance(v, dict) else None)
        for v in obj.values():
            r = find_hash(v, fname)
            if r:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_hash(v, fname)
            if r:
                return r
    return None

ART = json.loads(locate("artifact_hashes.json", [P3]).read_text())
for fn in ("protocol_D.csv", "protocol_D_inner.csv"):
    want = find_hash(ART, fn)
    got = sha256_file(SPL / fn)
    check(f"split hash {fn}", "WARN" if want is None else (got == want),
          "no entry in artifact_hashes.json" if want is None else got[:16])

[PASS] gate sha256 == gate_hash.txt (start) 40df2fb82096e994
[PASS] gate sha256 == frozen Phase 4 hash (start) 
[PASS] split hash protocol_D.csv deaba544ec4dec54
[PASS] split hash protocol_D_inner.csv 42a0c69a6db36ad4


In [3]:
# ---- src/soh_deep_v2.py is owned by this notebook: embedded source, hash, overwrite if different
MODULE_SRC = r'''"""soh_deep_v2: deep models and batched trainer for Phase 5a2.

OWNED by notebook 05a2_deep_pilot_fix.ipynb. The notebook overwrites this file
whenever its SHA-256 differs from the embedded source. Do not edit by hand.
"""
import math
import time
import hashlib
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MODULE_HASH_OK = True
KINDS = ("M0", "M1", "M1p")


def self_hash():
    """SHA-256 of the source file this module was loaded from."""
    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()


def sub_idx(L):
    """32 grid points used by M0 and by the tree comparators."""
    return np.round(np.linspace(0, L - 1, 32)).astype(np.int64)


# ---------------------------------------------------------------- single models
class SingleM0(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.register_buffer("idx", torch.as_tensor(sub_idx(L)), persistent=False)
        self.fc1 = nn.Linear(33, 44)
        self.fc2 = nn.Linear(44, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, d, q):
        x = torch.cat([d[:, self.idx], q[:, None]], 1)
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc3(h).squeeze(1)


class SingleM1(nn.Module):
    def __init__(self, L, pos=False):
        super().__init__()
        self.pos = bool(pos)
        c = 2 if pos else 1
        self.conv1 = nn.Conv1d(c, 8, 9, 2, 4)
        self.bn1 = nn.BatchNorm1d(8)
        self.conv2 = nn.Conv1d(8, 16, 5, 2, 2)
        self.bn2 = nn.BatchNorm1d(16)
        self.conv3 = nn.Conv1d(16, 32, 3, 2, 1)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(33, 16)
        self.fc2 = nn.Linear(16, 1)
        if pos:
            self.register_buffer("posch", torch.linspace(-1, 1, L), persistent=False)

    def forward(self, d, q):
        x = d[:, None, :]
        if self.pos:
            p = self.posch.view(1, 1, -1).expand(d.shape[0], 1, -1)
            x = torch.cat([x, p], 1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = x.mean(-1)
        h = torch.cat([x, q[:, None]], 1)
        h = F.relu(self.fc1(h))
        return self.fc2(h).squeeze(1)


def make_single(kind, L, seed, dtype=torch.float32):
    torch.manual_seed(int(seed))
    if kind == "M0":
        m = SingleM0(L)
    elif kind == "M1":
        m = SingleM1(L, False)
    elif kind == "M1p":
        m = SingleM1(L, True)
    else:
        raise ValueError("unknown model kind %r" % (kind,))
    return m.to(dtype)


def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))


# -------------------------------------------------------------- batched models
class BatchedM0(nn.Module):
    """G independent M0 members; per-member linear layers via einsum."""

    def __init__(self, singles, L):
        super().__init__()
        self.G = len(singles)
        self.register_buffer("idx", torch.as_tensor(sub_idx(L)), persistent=False)

        def W(name):
            return nn.Parameter(torch.stack(
                [getattr(s, name).weight.detach().t().clone() for s in singles]))

        def b(name):
            return nn.Parameter(torch.stack(
                [getattr(s, name).bias.detach().clone() for s in singles]))

        self.W1, self.b1 = W("fc1"), b("fc1")
        self.W2, self.b2 = W("fc2"), b("fc2")
        self.W3, self.b3 = W("fc3"), b("fc3")

    def forward(self, d, q):
        x = torch.cat([d[:, :, self.idx], q[:, :, None]], 2)
        h = F.relu(torch.einsum("gbi,gio->gbo", x, self.W1) + self.b1[:, None, :])
        h = F.relu(torch.einsum("gbi,gio->gbo", h, self.W2) + self.b2[:, None, :])
        return (torch.einsum("gbi,gio->gbo", h, self.W3) + self.b3[:, None, :]).squeeze(2)

    def member_params(self, g):
        return [self.W1[g].t(), self.b1[g], self.W2[g].t(), self.b2[g],
                self.W3[g].t(), self.b3[g]]


class BatchedM1(nn.Module):
    """G independent M1 / M1p members: grouped Conv1d, per-member BN channels."""

    def __init__(self, singles, L, pos=False):
        super().__init__()
        G = len(singles)
        self.G, self.pos = G, bool(pos)
        c = 2 if pos else 1
        dt = next(singles[0].parameters()).dtype
        self.conv1 = nn.Conv1d(G * c, G * 8, 9, 2, 4, groups=G)
        self.bn1 = nn.BatchNorm1d(G * 8)
        self.conv2 = nn.Conv1d(G * 8, G * 16, 5, 2, 2, groups=G)
        self.bn2 = nn.BatchNorm1d(G * 16)
        self.conv3 = nn.Conv1d(G * 16, G * 32, 3, 2, 1, groups=G)
        self.bn3 = nn.BatchNorm1d(G * 32)
        self.W1 = nn.Parameter(torch.zeros(G, 33, 16))
        self.b1 = nn.Parameter(torch.zeros(G, 16))
        self.W2 = nn.Parameter(torch.zeros(G, 16, 1))
        self.b2 = nn.Parameter(torch.zeros(G, 1))
        if pos:
            self.register_buffer("posch", torch.linspace(-1, 1, L), persistent=False)
        self.to(dt)
        with torch.no_grad():
            for g, s in enumerate(singles):
                for i in ("1", "2", "3"):
                    conv, bn = getattr(self, "conv" + i), getattr(self, "bn" + i)
                    sc, sb = getattr(s, "conv" + i), getattr(s, "bn" + i)
                    o = conv.out_channels // G
                    sl = slice(g * o, (g + 1) * o)
                    conv.weight[sl] = sc.weight
                    conv.bias[sl] = sc.bias
                    bn.weight[sl] = sb.weight
                    bn.bias[sl] = sb.bias
                self.W1[g] = s.fc1.weight.t()
                self.b1[g] = s.fc1.bias
                self.W2[g] = s.fc2.weight.t()
                self.b2[g] = s.fc2.bias

    def forward(self, d, q):
        G, B, L = d.shape
        if self.pos:
            p = self.posch.view(1, 1, L).expand(G, B, L)
            x = torch.stack([d, p], 2).permute(1, 0, 2, 3).reshape(B, G * 2, L)
        else:
            x = d.permute(1, 0, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = x.mean(-1).reshape(B, G, 32).permute(1, 0, 2)
        h = torch.cat([x, q[:, :, None]], 2)
        h = F.relu(torch.einsum("gbi,gio->gbo", h, self.W1) + self.b1[:, None, :])
        return (torch.einsum("gbi,gio->gbo", h, self.W2) + self.b2[:, None, :]).squeeze(2)

    def member_params(self, g):
        out = []
        for i in ("1", "2", "3"):
            conv, bn = getattr(self, "conv" + i), getattr(self, "bn" + i)
            o = conv.out_channels // self.G
            sl = slice(g * o, (g + 1) * o)
            out += [conv.weight[sl], conv.bias[sl], bn.weight[sl], bn.bias[sl]]
        out += [self.W1[g].t(), self.b1[g], self.W2[g].t(), self.b2[g]]
        return out


def make_batched(kind, L, singles):
    if kind == "M0":
        return BatchedM0(singles, L)
    if kind == "M1":
        return BatchedM1(singles, L, False)
    if kind == "M1p":
        return BatchedM1(singles, L, True)
    raise ValueError("unknown model kind %r" % (kind,))


# ----------------------------------------------------------------- scaling etc.
def fit_scalers(X, q0, y, rows_list, head, nominal, device, dtype):
    """Per-member scalers fitted on that member's training rows only."""
    s, qm, qs, ym, ys = [], [], [], [], []
    for r in rows_list:
        rt = torch.as_tensor(np.asarray(r), device=device)
        s.append(float(X[rt].double().std()))
        qv = q0[rt].double()
        qm.append(float(qv.mean()))
        qs.append(float(qv.std()))
        if head == "abs":
            yv = y[rt].double()
            ym.append(float(yv.mean()))
            ys.append(float(yv.std()))
        elif head == "res":
            yv = (y[rt] - 100.0 * q0[rt] / nominal).double()
            ym.append(0.0)
            ys.append(float(yv.std()))
        else:
            raise ValueError("unknown head %r" % (head,))
    out = {}
    for name, v in (("s", s), ("qs", qs), ("ys", ys)):
        out[name] = torch.tensor([x if x > 1e-12 else 1.0 for x in v],
                                 dtype=dtype, device=device)
    out["qm"] = torch.tensor(qm, dtype=dtype, device=device)
    out["ym"] = torch.tensor(ym, dtype=dtype, device=device)
    return out


def make_inputs(X, q0, idx, sc):
    d = X[idx] / sc["s"][:, None, None]
    q = (q0[idx] - sc["qm"][:, None]) / sc["qs"][:, None]
    return d, q


def base_pp(q0, idx, sc, head, nominal):
    if head == "abs":
        return sc["ym"][:, None].expand(idx.shape)
    return 100.0 * q0[idx] / nominal


def lr_at(it, total, warm, lr):
    if it < warm:
        return lr * (it + 1) / warm
    prog = (it - warm) / max(1, total - warm)
    return 0.5 * lr * (1.0 + math.cos(math.pi * min(1.0, prog)))


def clip_per_member(params, G, clip):
    sq = None
    for p in params:
        if not p.grad.is_contiguous():
            p.grad = p.grad.contiguous()
        s = p.grad.view(G, -1).pow(2).sum(1)
        sq = s if sq is None else sq + s
    coef = (clip / (sq.sqrt() + 1e-6)).clamp(max=1.0)
    for p in params:
        p.grad.view(G, -1).mul_(coef[:, None])


def epoch_positions(rng, n, need):
    parts, tot = [], 0
    while tot < need:
        parts.append(rng.permutation(n))
        tot += n
    return np.concatenate(parts)[:need]


def cell_macro_mae(pred, yv, codes):
    """Mean over cells of the per-cell mean absolute error. Returns (macro, per_cell)."""
    err = np.abs(np.asarray(pred, dtype=np.float64) - np.asarray(yv, dtype=np.float64))
    k = int(codes.max()) + 1
    s = np.bincount(codes, weights=err, minlength=k)
    c = np.bincount(codes, minlength=k)
    ok = c > 0
    per = s[ok] / c[ok]
    return float(per.mean()), per


@torch.no_grad()
def predict(model, X, q0, rows_list, sc, head, nominal, device, dtype, chunk=1024):
    """Eval-mode predictions in pp for every member; rows_list may differ in length."""
    model.eval()
    G = len(rows_list)
    n = [len(r) for r in rows_list]
    M = max(n)
    pad = np.zeros((G, M), dtype=np.int64)
    for g, r in enumerate(rows_list):
        if n[g]:
            pad[g, :n[g]] = r
            pad[g, n[g]:] = r[0]
    pt = torch.from_numpy(pad).to(device)
    out = torch.empty((G, M), dtype=dtype, device=device)
    for a in range(0, M, chunk):
        idx = pt[:, a:a + chunk]
        d, q = make_inputs(X, q0, idx, sc)
        z = model(d, q)
        out[:, a:a + chunk] = base_pp(q0, idx, sc, head, nominal) + sc["ys"][:, None] * z
    out = out.cpu().numpy()
    return [out[g, :n[g]] for g in range(G)]


# --------------------------------------------------------------------- trainer
def train_members(X, q0, y, members, kind, head, lr, epochs, cfg, device,
                  dtype=torch.float32, log_every=10):
    """Train G members at once. members: dicts with seed, train_rows, train_w,
    val_rows, val_codes. X (N,L), q0 (N,), y (N,) are tensors on `device`."""
    t0 = time.perf_counter()
    G, L = len(members), X.shape[1]
    B, nominal, sigma = cfg["BATCH"], cfg["NOMINAL_AH"], cfg["NOISE_SIGMA"]
    singles = [make_single(kind, L, m["seed"], dtype) for m in members]
    model = make_batched(kind, L, singles).to(device)
    del singles
    train_rows = [np.asarray(m["train_rows"], dtype=np.int64) for m in members]
    val_rows = [np.asarray(m["val_rows"], dtype=np.int64) for m in members]
    sc = fit_scalers(X, q0, y, train_rows, head, nominal, device, dtype)
    n = [len(r) for r in train_rows]
    steps = int(math.ceil(max(n) / B))
    total = epochs * steps
    warm = max(1, int(round(cfg["WARMUP_FRAC"] * total)))
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=cfg["WD"])
    gen = torch.Generator(device=device)
    gen.manual_seed(int(members[0]["seed"]) * 7919 + G)
    rngs = [np.random.default_rng([int(m["seed"]), 991]) for m in members]
    y_cpu = y.detach().cpu().numpy().astype(np.float64)
    ep_pts, val_curve, loss_curve = [], [], []
    it = 0
    for ep in range(1, epochs + 1):
        model.train()
        rows_ep = np.empty((G, steps * B), dtype=np.int64)
        w_ep = np.empty((G, steps * B), dtype=np.float64)
        for g, m in enumerate(members):
            pos = epoch_positions(rngs[g], n[g], steps * B)
            rows_ep[g] = train_rows[g][pos]
            w_ep[g] = np.asarray(m["train_w"], dtype=np.float64)[pos]
        rows_t = torch.from_numpy(rows_ep).to(device)
        w_t = torch.from_numpy(w_ep).to(device=device, dtype=dtype)
        loss_acc = torch.zeros(G, device=device, dtype=dtype)
        for s in range(steps):
            lr_now = lr_at(it, total, warm, lr)
            for pg in opt.param_groups:
                pg["lr"] = lr_now
            idx = rows_t[:, s * B:(s + 1) * B]
            w = w_t[:, s * B:(s + 1) * B]
            d, q = make_inputs(X, q0, idx, sc)
            if sigma > 0:
                d = d + sigma * torch.randn(d.shape, device=device, dtype=dtype, generator=gen)
            z = model(d, q)
            pred = base_pp(q0, idx, sc, head, nominal) + sc["ys"][:, None] * z
            l = F.huber_loss(pred, y[idx], delta=cfg["HUBER_DELTA"], reduction="none")
            lg = (w * l).sum(1) / w.sum(1)
            opt.zero_grad(set_to_none=True)
            lg.sum().backward()
            clip_per_member(params, G, cfg["CLIP"])
            opt.step()
            loss_acc += lg.detach()
            it += 1
        if ep % log_every == 0 or ep == epochs:
            vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype)
            vm = [cell_macro_mae(vp[g], y_cpu[val_rows[g]], members[g]["val_codes"])[0]
                  for g in range(G)]
            ep_pts.append(ep)
            val_curve.append(vm)
            loss_curve.append((loss_acc / steps).cpu().numpy().astype(np.float64).tolist())
    vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype)
    tp = predict(model, X, q0, train_rows, sc, head, nominal, device, dtype)
    train_mae = []
    for g in range(G):
        w = np.asarray(members[g]["train_w"], dtype=np.float64)
        train_mae.append(float((w * np.abs(tp[g].astype(np.float64) - y_cpu[train_rows[g]])).sum() / w.sum()))
    if device.type == "cuda":
        torch.cuda.synchronize()
    return dict(
        val_pred=[v.astype(np.float32) for v in vp],
        train_mae=np.array(train_mae),
        curve=dict(epochs=ep_pts, val_mae=np.array(val_curve), train_loss=np.array(loss_curve)),
        seconds=time.perf_counter() - t0,
        steps_per_epoch=steps,
        n_members=G,
        nan_members=[g for g in range(G) if not np.isfinite(vp[g]).all()],
    )


# ---------------------------------------------------------------- verification
def parity_check(kind, X, q0, y, rows_list, seeds, head, cfg, device, dtype,
                 steps=20, lr=2e-3, B=64):
    """Batched members vs ordinary single models: same init, same 20 steps."""
    G, L = len(seeds), X.shape[1]
    nominal = cfg["NOMINAL_AH"]
    sing_a = [make_single(kind, L, s, dtype) for s in seeds]
    sing_b = [make_single(kind, L, s, dtype).to(device) for s in seeds]
    bm = make_batched(kind, L, sing_a).to(device)
    sc = fit_scalers(X, q0, y, rows_list, head, nominal, device, dtype)
    rng = np.random.default_rng(0)
    idx_all = np.stack([np.stack([rng.choice(np.asarray(r), B) for r in rows_list])
                        for _ in range(steps)])
    w_all = rng.uniform(0.5, 1.5, (steps, G, B))
    opt_b = torch.optim.AdamW(bm.parameters(), lr=lr, weight_decay=cfg["WD"])
    opts = [torch.optim.AdamW(s.parameters(), lr=lr, weight_decay=cfg["WD"]) for s in sing_b]
    params = list(bm.parameters())
    bm.train()
    for s in sing_b:
        s.train()
    for t in range(steps):
        idx = torch.from_numpy(idx_all[t]).to(device)
        w = torch.from_numpy(w_all[t]).to(device=device, dtype=dtype)
        d, q = make_inputs(X, q0, idx, sc)
        pred = base_pp(q0, idx, sc, head, nominal) + sc["ys"][:, None] * bm(d, q)
        l = F.huber_loss(pred, y[idx], delta=cfg["HUBER_DELTA"], reduction="none")
        lg = (w * l).sum(1) / w.sum(1)
        opt_b.zero_grad(set_to_none=True)
        lg.sum().backward()
        clip_per_member(params, G, cfg["CLIP"])
        opt_b.step()
        for g in range(G):
            ig, wg = idx[g], w[g]
            dg = X[ig] / sc["s"][g]
            qg = (q0[ig] - sc["qm"][g]) / sc["qs"][g]
            bg = sc["ym"][g] if head == "abs" else 100.0 * q0[ig] / nominal
            pg = bg + sc["ys"][g] * sing_b[g](dg, qg)
            lo = F.huber_loss(pg, y[ig], delta=cfg["HUBER_DELTA"], reduction="none")
            loss = (wg * lo).sum() / wg.sum()
            opts[g].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sing_b[g].parameters(), cfg["CLIP"])
            opts[g].step()
    max_p = 0.0
    for g in range(G):
        for a, b in zip(bm.member_params(g), sing_b[g].parameters()):
            max_p = max(max_p, float((a.detach() - b.detach()).abs().max()))
    ev_rows = [np.asarray(r)[:200] for r in rows_list]
    pb = predict(bm, X, q0, ev_rows, sc, head, nominal, device, dtype)
    max_e = 0.0
    with torch.no_grad():
        for g in range(G):
            sing_b[g].eval()
            ig = torch.as_tensor(ev_rows[g], device=device)
            dg = X[ig] / sc["s"][g]
            qg = (q0[ig] - sc["qm"][g]) / sc["qs"][g]
            bg = sc["ym"][g] if head == "abs" else 100.0 * q0[ig] / nominal
            ps = (bg + sc["ys"][g] * sing_b[g](dg, qg)).cpu().numpy()
            max_e = max(max_e, float(np.abs(ps - pb[g]).max()))
    return dict(kind=kind, dtype=str(dtype), max_param_diff=max_p, max_eval_diff=max_e)


def same_seed_identical(kind, L, seed):
    a = make_single(kind, L, seed)
    b = make_single(kind, L, seed)
    return all(torch.equal(p, r) for p, r in zip(a.parameters(), b.parameters()))
'''
MODULE_BYTES = MODULE_SRC.encode("utf-8")
MODULE_HASH = hashlib.sha256(MODULE_BYTES).hexdigest()
MOD_PATH = SRC / "soh_deep_v2.py"
SRC.mkdir(exist_ok=True)
if not MOD_PATH.exists() or sha256_file(MOD_PATH) != MODULE_HASH:
    MOD_PATH.write_bytes(MODULE_BYTES)
    print("soh_deep_v2.py written (disk copy was missing or different)")
sys.modules.pop("soh_deep_v2", None)
importlib.invalidate_caches()
import soh_deep_v2 as mod
check("module hash on disk == embedded", sha256_file(MOD_PATH) == MODULE_HASH, MODULE_HASH)
check("module imported from disk copy", mod.self_hash() == MODULE_HASH and mod.MODULE_HASH_OK is True)
print("MODULE_HASH", MODULE_HASH)

# ---- GPU, RAM
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
check("CUDA available", torch.cuda.is_available(), "running on CPU would be far too slow" if DEVICE.type == "cpu" else "")
if DEVICE.type == "cuda":
    pr = torch.cuda.get_device_properties(0)
    GPU_INFO = f"{pr.name}, {pr.total_memory / 2**30:.1f} GB, cc {pr.major}.{pr.minor}, torch {torch.__version__}, cuda {torch.version.cuda}"
else:
    GPU_INFO = f"cpu only, torch {torch.__version__}"
GPU_INFO += f", TF32 matmul={torch.backends.cuda.matmul.allow_tf32} cudnn={torch.backends.cudnn.allow_tf32}"
print(GPU_INFO)
avail_gb = psutil.virtual_memory().available / 2**30
procs = []
for p in psutil.process_iter(["name", "memory_info"]):
    try:
        procs.append((p.info["memory_info"].rss / 2**30, p.info["name"]))
    except Exception:
        pass
TOP_PROCS = sorted(procs, reverse=True)[:8]
print(f"available RAM {avail_gb:.1f} GB; top processes:", ", ".join(f"{n} {g:.2f}GB" for g, n in TOP_PROCS))
if avail_gb < 2:
    check("RAM available >= 2 GB", "WARN", f"{avail_gb:.1f} GB; close the largest apps listed above (browser, IDE, other Python kernels)")
else:
    check("RAM available >= 2 GB", True, f"{avail_gb:.1f} GB")
RAM_INFO = f"{avail_gb:.1f} GB available; top: " + ", ".join(f"{n} {g:.2f}GB" for g, n in TOP_PROCS)

[PASS] module hash on disk == embedded fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6
[PASS] module imported from disk copy 
MODULE_HASH fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6
[PASS] CUDA available 
NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6.0 GB, cc 8.6, torch 2.14.1+cu130, cuda 13.0, TF32 matmul=False cudnn=False
available RAM 2.6 GB; top processes: Code.exe 1.76GB, Code.exe 1.56GB, Code.exe 0.78GB, python.exe 0.64GB, explorer.exe 0.30GB, chrome.exe 0.29GB, MsMpEng.exe 0.27GB, chrome.exe 0.26GB
[PASS] RAM available >= 2 GB 2.6 GB


## 2. Data and verification

In [4]:
import soh_common as sc
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import sklearn

def need_cols(df, cols, where):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"{where}: missing column(s) {miss}; columns present: {list(df.columns)}")

_DATA = {}

def load_data():
    """Load tables once; build development-row arrays. No outer-test selection happens here."""
    if _DATA:
        return _DATA
    cycles, qdlin, t_knots, registry = sc.load_tables(root=ROOT, mmap=True)
    labels = pd.read_parquet(PROC / "labels_table_v2.parquet")
    need_cols(labels, ["row_id", "cell_id", "batch", "cycle_number", "q0", "soh_nom", "scored2"], "labels_table_v2")
    need_cols(cycles, ["row_id"], "cycles_table")
    check("labels rows == 104971", len(labels) == 104971, len(labels))
    check("qdlin shape (104971, 1000) float32", qdlin.shape == (104971, 1000) and qdlin.dtype == np.float32, f"{qdlin.shape} {qdlin.dtype}")
    check("cycles row_id == row position == qdlin row", np.array_equal(np.asarray(cycles["row_id"]), np.arange(len(cycles))) and len(cycles) == qdlin.shape[0])
    check("labels row_id unique and within qdlin", labels["row_id"].is_unique and labels["row_id"].max() < qdlin.shape[0])

    qref = np.load(locate("qref_f32.npy", [PROC, P3]), mmap_mode="r")
    qids = pd.read_csv(locate("qref_cell_ids.csv", [PROC, P3]))
    need_cols(qids, ["row", "cell_id"], "qref_cell_ids.csv")
    check("qref shape (129, 1000)", qref.shape == (129, 1000), qref.shape)
    check("qref_cell_ids row == position", np.array_equal(qids["row"].values, np.arange(len(qids))))
    qref_row = dict(zip(qids["cell_id"], qids["row"]))

    pdD = pd.read_csv(SPL / "protocol_D.csv")
    pdI = pd.read_csv(SPL / "protocol_D_inner.csv")
    need_cols(pdD, ["repeat", "outer_fold", "cell_id", "role", "seed", "tertile"], "protocol_D.csv")
    need_cols(pdI, ["repeat", "outer_fold", "inner_fold", "cell_id", "role"], "protocol_D_inner.csv")
    check("protocol_D roles == {train, test}", set(pdD["role"]) == {"train", "test"}, set(pdD["role"]))
    roles = set(pdI["role"])
    check("protocol_D_inner has exactly two role values incl. 'train'", len(roles) == 2 and "train" in roles, roles)

    dev_cells = sorted(pdD["cell_id"].unique())
    n1 = sum(c.startswith("b1") for c in dev_cells); n2 = sum(c.startswith("b2") for c in dev_cells)
    check("development cells: 84 (41 b1, 43 b2)", True if (len(dev_cells), n1, n2) == (84, 41, 43) else "WARN", f"{len(dev_cells)} ({n1} b1, {n2} b2)")
    bad = [c for c in dev_cells if c in SENSITIVITY or not (c.startswith("b1") or c.startswith("b2"))]
    check("no sensitivity or batch-3 cell among development cells", not bad, bad)
    dev_cells = [c for c in dev_cells if c not in bad]
    miss = [c for c in dev_cells if c not in qref_row]
    check("every development cell has a qref row", not miss, miss)

    dev = labels[labels["scored2"].astype(bool) & labels["cell_id"].isin(dev_cells)]
    dev = dev.sort_values(["cell_id", "cycle_number"], kind="stable").reset_index(drop=True)
    check("development rows == 55963", True if len(dev) == 55963 else "WARN", len(dev))
    for c in ("q0", "soh_nom", "cycle_number"):
        check(f"finite {c} on development rows", bool(np.isfinite(dev[c].values.astype(float)).all()))
    cells_arr = dev["cell_id"].values
    cell_list = sorted(set(cells_arr))
    rows_by_cell, code = {}, np.empty(len(dev), np.int32)
    for k, c in enumerate(cell_list):
        ix = np.flatnonzero(cells_arr == c)
        rows_by_cell[c] = ix
        code[ix] = k
    _DATA.update(dict(
        qdlin=qdlin, qref=qref, qref_row=qref_row, pdD=pdD, pdI=pdI, dev_cells=dev_cells,
        row_ids=dev["row_id"].values.astype(np.int64), cell_arr=cells_arr, cell_list=cell_list,
        cell_code=code, rows_by_cell=rows_by_cell,
        q0=dev["q0"].values.astype(np.float32), y=(100.0 * dev["soh_nom"].values.astype(np.float64)),
        cycle=dev["cycle_number"].values.astype(np.float64)))
    return _DATA

def window_geom(name):
    ov, cv = WINDOW_V[name]
    sl = sc.window_slice(ov, cv)
    io, ic = sl.start, sl.stop - 1
    return io, ic, ic - io + 1

def build_dq(cfg, D, name):
    """Delta-Q for all development rows, built here (never from the Phase 4 cache), cached by window."""
    io, ic, L = window_geom(name)
    p, pr = cfg["cache"] / f"dq_{name}.npy", cfg["cache"] / f"dq_{name}_rowids.npy"
    if p.exists() and pr.exists() and np.array_equal(np.load(pr), D["row_ids"]):
        return np.load(p)
    out = np.empty((len(D["row_ids"]), L), np.float32)
    for c, ix in D["rows_by_cell"].items():
        block = np.asarray(D["qdlin"][D["row_ids"][ix]])
        qref = np.asarray(D["qref"][D["qref_row"][c]])
        r = np.asarray(sc.build_delta_q(block, qref, io, ic), dtype=np.float32)
        if r.shape != (len(ix), L):
            raise ValueError(f"build_delta_q returned {r.shape} for cell {c}, expected {(len(ix), L)}")
        out[ix] = r
    np.save(p, out); np.save(pr, D["row_ids"])
    return out

In [5]:
# ---- inner problems and member recipe
def build_problems(cfg, D):
    pdD, pdI = D["pdD"], D["pdI"]
    devset = set(D["dev_cells"])
    probs = []
    for oi, (rep, of) in enumerate(cfg["OUTER_SETS"]):
        a = pdD[(pdD["repeat"] == rep) & (pdD["outer_fold"] == of)]
        tr = sorted(c for c in a[a["role"] == "train"]["cell_id"].unique() if c in devset)
        te = sorted(a[a["role"] == "test"]["cell_id"].unique())
        check(f"outer set {(rep, of)}: 13-17 outer-test cells", 13 <= len(te) <= 17, f"{len(te)} (train {len(tr)})")
        check(f"outer set {(rep, of)}: train/test disjoint", not (set(tr) & set(te)))
        b = pdI[(pdI["repeat"] == rep) & (pdI["outer_fold"] == of)]
        folds = sorted(b["inner_fold"].unique())
        check(f"outer set {(rep, of)}: 4 inner folds", len(folds) == 4, folds)
        allval = []
        for f in folds:
            bf = b[b["inner_fold"] == f]
            val = sorted(bf[bf["role"] != "train"]["cell_id"]); trn = sorted(bf[bf["role"] == "train"]["cell_id"])
            allval += val
            probs.append(dict(pid=len(probs), oi=oi, rep=int(rep), of=int(of), fold=int(f),
                              train_cells=trn, val_cells=val, test_cells=te, outer_train=tr))
        check(f"outer set {(rep, of)}: inner validation folds partition the outer-train cells",
              sorted(allval) == tr and len(set(allval)) == len(allval), f"{len(allval)} vs {len(tr)}")
    return probs

def make_member(D, p, seed, cfg):
    """Strided rows (every 5th, random per-cell offset), weights 1/(rows of cell); val = all rows of val cells."""
    rng = np.random.default_rng([int(seed), p["rep"], p["of"], p["fold"], 17])
    rows, w = [], []
    for c in p["train_cells"]:
        ix = D["rows_by_cell"][c]
        off = int(rng.integers(0, cfg["TRAIN_STRIDE"]))
        sel = ix[off::cfg["TRAIN_STRIDE"]]
        if len(sel):
            rows.append(sel); w.append(np.full(len(sel), 1.0 / len(sel)))
    vr = [D["rows_by_cell"][c] for c in p["val_cells"]]
    return dict(seed=int(seed), pid=p["pid"], train_rows=np.concatenate(rows), train_w=np.concatenate(w),
                val_rows=np.concatenate(vr),
                val_codes=np.concatenate([np.full(len(v), k, np.int32) for k, v in enumerate(vr)]))

def assert_no_leak(D, p, m):
    """Members may only touch outer-train cells; validation cells never overlap training cells."""
    cl = D["cell_list"]
    tcells = {cl[k] for k in np.unique(D["cell_code"][m["train_rows"]])}
    vcells = {cl[k] for k in np.unique(D["cell_code"][m["val_rows"]])}
    ok = (tcells <= set(p["outer_train"]) and vcells <= set(p["outer_train"]) and not (tcells & vcells)
          and not ((tcells | vcells) & (set(p["test_cells"]) | SENSITIVITY)))
    if not ok:
        raise RuntimeError(f"leakage guard failed for problem {p['pid']} seed {m['seed']}")

# ---- verification (2a-2d)
def verify(ctx):
    cfg, D, probs = ctx["cfg"], ctx["D"], ctx["problems"]
    V = dict(dq=[], allowed=None, parity=[], counts=[], seed=[])
    rng = np.random.default_rng(cfg["SEED"])
    # window geometry
    for w, (o, c) in WINDOW_IDX.items():
        io, ic, L = window_geom(w)
        check(f"{w} geometry", (io, ic) == (o, c) and L == c - o + 1 and sc.grid_index(WINDOW_V[w][0]) == 0
              and sc.grid_index(WINDOW_V[w][1]) == c, f"i_open={io} i_close={ic} width={L}")
    # 2a
    for w in cfg["WINDOWS"]:
        io, ic, L = window_geom(w)
        DQ = build_dq(cfg, D, w)
        cells = list(rng.choice(D["dev_cells"], 3, replace=False))
        md = 0.0
        for c in cells:
            ix = D["rows_by_cell"][c]
            for i in ix:
                one = np.asarray(sc.build_delta_q(np.asarray(D["qdlin"][D["row_ids"][i]]),
                                                  np.asarray(D["qref"][D["qref_row"][c]]), io, ic))
                md = max(md, float(np.abs(one - DQ[i]).max()))
        z = float(np.abs(DQ[:, 0]).max())
        fin = bool(np.isfinite(DQ).all())
        V["dq"].append(dict(window=w, cells=[str(c) for c in cells], max_abs_diff=md, max_abs_first=z, width=int(DQ.shape[1]), finite=fin))
        check(f"{w} delta-Q block vs single-vector (3 cells)", md < 1e-6, f"max diff {md:.2e}")
        check(f"{w} delta-Q[0]==0, length==width, finite", z == 0.0 and DQ.shape[1] == L and fin, f"first={z} len={DQ.shape[1]} finite={fin}")
        del DQ
    # 2b
    try:
        sc.assert_allowed(["q0"], roles=("primary",)); roles = ["primary"]
    except Exception:
        sc.assert_allowed(["q0"], roles=("primary", "ablation")); roles = ["primary", "ablation"]
    V["allowed"] = roles
    check("assert_allowed(['q0'])", True, f"passed with roles {roles}")
    print("Delta-Q values are derived from Qdlin at or before the window close (as in Phase 4, where derived columns were checked through q0).")
    # 2c
    w0 = cfg["WINDOWS"][0]
    DQ = build_dq(cfg, D, w0)
    p = probs[0]
    seeds = (42, 43, 44)
    mems = [make_member(D, p, s, cfg) for s in seeds]
    uni = np.unique(np.concatenate([m["train_rows"] for m in mems]))
    loc = [np.searchsorted(uni, m["train_rows"]) for m in mems]
    for dtype, tol in ((torch.float64, 1e-8), (torch.float32, 1e-4)):
        X = torch.as_tensor(DQ[uni], dtype=dtype, device=DEVICE)
        q0 = torch.as_tensor(D["q0"][uni], dtype=dtype, device=DEVICE)
        y = torch.as_tensor(D["y"][uni], dtype=dtype, device=DEVICE)
        for kind in KINDS:
            try:
                r = mod.parity_check(kind, X, q0, y, loc, seeds, "abs", cfg, DEVICE, dtype)
                V["parity"].append(r)
                ok = r["max_param_diff"] < tol and r["max_eval_diff"] < tol
                check(f"parity {kind} {str(dtype).split('.')[-1]}", True if ok else ("WARN" if dtype == torch.float32 else False),
                      f"param {r['max_param_diff']:.2e} eval {r['max_eval_diff']:.2e}")
            except Exception as e:
                record_error(f"parity {kind} {dtype}", e)
    del DQ, X, q0, y
    # 2d
    exp = {"M0": 2969, "M1": 2977, "M1p": 3049}
    for kind in KINDS:
        n = mod.count_params(mod.make_single(kind, 334, 42))
        n2 = mod.count_params(mod.make_single(kind, 734, 42))
        nb = mod.count_params(mod.make_batched(kind, 334, [mod.make_single(kind, 334, s) for s in (1, 2, 3)]))
        V["counts"].append(dict(kind=kind, params=n, expected=exp[kind], batched3=nb))
        check(f"param count {kind}", n == exp[kind] and n2 == exp[kind] and nb == 3 * exp[kind], f"{n} (expected {exp[kind]}), batched G=3: {nb}")
    DQ = build_dq(cfg, D, cfg["WINDOWS"][0])
    xs = torch.as_tensor(DQ[:64]); qs = torch.as_tensor(D["q0"][:64])
    for kind in KINDS:
        a, b = mod.make_single(kind, DQ.shape[1], 42).eval(), mod.make_single(kind, DQ.shape[1], 42).eval()
        with torch.no_grad():
            same = bool(torch.equal(a(xs, qs), b(xs, qs))) and mod.same_seed_identical(kind, DQ.shape[1], 42)
            diff = not torch.equal(a(xs, qs), mod.make_single(kind, DQ.shape[1], 43).eval()(xs, qs))
        V["seed"].append(dict(kind=kind, identical=same, differs_other_seed=diff))
        check(f"same seed identical predictions {kind}", same and diff)
    ctx["verif"] = V

## 3. Pilot-2 (inner folds only)

In [6]:
def key_str(k):
    return json.dumps(k, sort_keys=True)

def run_key(cfg, window, model, head, lr):
    return dict(module=MODULE_HASH, window=window, model=model, head=head, lr=lr, seeds=list(cfg["SEEDS"]),
                E=cfg["E"], cfg=cfg["name"], outer_sets=[list(x) for x in cfg["OUTER_SETS"]],
                hp={k: cfg[k] for k in ("BATCH", "CLIP", "HUBER_DELTA", "WD", "NOISE_SIGMA", "WARMUP_FRAC", "TRAIN_STRIDE", "NOMINAL_AH", "SEED")})

def run_path(cfg, k):
    return cfg["cache"] / ("run_" + hashlib.sha256(key_str(k).encode()).hexdigest()[:20] + ".npz")

def save_run(path, k, r):
    lens = np.array([len(v) for v in r["val_pred"]])
    np.savez(path, key=np.array(key_str(k)), pred=np.concatenate(r["val_pred"]), lens=lens, train_mae=r["train_mae"],
             ce=np.array(r["curve"]["epochs"]), cv=r["curve"]["val_mae"], cl=r["curve"]["train_loss"],
             seconds=np.array(r["seconds"]), n_members=np.array(r["n_members"]), nan_members=np.array(r["nan_members"], dtype=int))

def load_run(path, k):
    if not path.exists():
        return None
    try:
        z = np.load(path, allow_pickle=False)
        if str(z["key"]) != key_str(k):
            return None
        cut = np.cumsum(z["lens"])[:-1]
        return dict(val_pred=np.split(z["pred"], cut), train_mae=z["train_mae"],
                    curve=dict(epochs=z["ce"].tolist(), val_mae=z["cv"], train_loss=z["cl"]),
                    seconds=float(z["seconds"]), n_members=int(z["n_members"]), nan_members=z["nan_members"].tolist())
    except Exception:
        return None

def run_pilot(ctx):
    cfg, D, probs = ctx["cfg"], ctx["D"], ctx["problems"]
    members, meta = [], []
    for seed in cfg["SEEDS"]:
        for p in probs:
            m = make_member(D, p, seed, cfg)
            assert_no_leak(D, p, m)
            members.append(m); meta.append((p["pid"], seed))
    ctx["meta"] = meta
    ctx["val_info"] = {}
    for p in probs:
        m0 = members[meta.index((p["pid"], cfg["SEEDS"][0]))]
        cells = np.array(p["val_cells"])
        ctx["val_info"][p["pid"]] = dict(rows=m0["val_rows"], codes=m0["val_codes"], y=D["y"][m0["val_rows"]],
                                         keep=~np.isin(cells[m0["val_codes"]], list(STITCHED)))
    runs = {}
    for window in cfg["WINDOWS"]:
        DQ = build_dq(cfg, D, window)
        X = torch.as_tensor(DQ, device=DEVICE)
        q0t = torch.as_tensor(D["q0"], device=DEVICE)
        yt = torch.as_tensor(D["y"].astype(np.float32), device=DEVICE)
        for model, head in VARIANTS:
            for lr in cfg["LRS"]:
                k = run_key(cfg, window, model, head, lr)
                path = run_path(cfg, k)
                name = f"{window} {model}-{head} lr={lr}"
                run = load_run(path, k)
                if run is None:
                    try:
                        run = mod.train_members(X, q0t, yt, members, model, head, lr, cfg["E"], cfg, DEVICE,
                                                torch.float32, cfg["LOG_EVERY"])
                        save_run(path, k, run)
                        log_run("deep:" + model, name, run["seconds"], False)
                    except Exception as e:
                        record_error("pilot " + name, e)
                        if DEVICE.type == "cuda":
                            torch.cuda.empty_cache()
                        continue
                else:
                    log_run("deep:" + model, name, 0.0, True)
                if run["nan_members"]:
                    check(f"non-finite predictions {name}", "WARN", f"members {run['nan_members']}")
                runs[(window, model, head, lr)] = run
        del X, q0t, yt, DQ
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    ctx["runs"] = runs

# ---- scoring
def macro(pred, yv, codes, keep=None):
    pred, yv, codes = np.asarray(pred, np.float64), np.asarray(yv), np.asarray(codes)
    if keep is not None:
        pred, yv, codes = pred[keep], yv[keep], codes[keep]
        _, codes = np.unique(codes, return_inverse=True)
    if codes.size == 0:
        return float("nan"), np.array([])
    return mod.cell_macro_mae(pred, yv, codes)

def se(a):
    a = np.asarray(a, float)
    return float(a.std(ddof=1) / np.sqrt(len(a))) if len(a) > 1 else float("nan")

def score_run(ctx, run):
    probs, meta, info = ctx["problems"], ctx["meta"], ctx["val_info"]
    n_out = len(ctx["cfg"]["OUTER_SETS"])
    S = {k: [] for k in ("single", "ens", "single_ns", "ens_ns", "train", "oi")}
    cellmae = []
    for p in probs:
        vi = info[p["pid"]]
        idx = [i for i, (pid, s) in enumerate(meta) if pid == p["pid"]]
        ps = [run["val_pred"][i] for i in idx]
        ens = np.mean(ps, axis=0)
        S["single"].append(np.mean([macro(q, vi["y"], vi["codes"])[0] for q in ps]))
        S["single_ns"].append(np.mean([macro(q, vi["y"], vi["codes"], vi["keep"])[0] for q in ps]))
        e, per = macro(ens, vi["y"], vi["codes"])
        S["ens"].append(e); cellmae.append(per)
        S["ens_ns"].append(macro(ens, vi["y"], vi["codes"], vi["keep"])[0])
        S["train"].append(float(np.mean(run["train_mae"][idx])))
        S["oi"].append(p["oi"])
    S = {k: np.array(v) for k, v in S.items()}
    outer = lambda a: [float(a[S["oi"] == k].mean()) for k in range(n_out)]
    cm = np.concatenate(cellmae)
    return dict(single_mae=float(S["single"].mean()), ens_mae=float(S["ens"].mean()), train_mae=float(S["train"].mean()),
                single_outer=outer(S["single"]), ens_outer=outer(S["ens"]),
                single_se=se(outer(S["single"])), ens_se=se(outer(S["ens"])),
                single_ns=float(S["single_ns"].mean()), ens_ns=float(S["ens_ns"].mean()),
                median_cell=float(np.median(cm)), p90_cell=float(np.percentile(cm, 90)))

def pilot_table(ctx):
    rows = []
    for (window, model, head, lr), run in ctx["runs"].items():
        r = dict(window=window, model=model, head=head, lr=lr, **score_run(ctx, run),
                 seconds=run["seconds"], n_members=run["n_members"], sec_per_member=run["seconds"] / run["n_members"])
        rows.append(r)
    ctx["pilot_df"] = pd.DataFrame(rows)

## 4. Comparators (exact Phase 4 definitions, same 12 problems and training-row recipe)

In [7]:
def comp_specs():
    s = []
    for d in (2, 3):
        for lr in (0.05, 0.1):
            s.append(("B0b", None, dict(max_depth=d, learning_rate=lr)))
    for comp in ("B3", "B3r"):
        for d in (2, 3, 4):
            for lr in (0.05, 0.1):
                s.append((comp, True, dict(max_depth=d, learning_rate=lr)))
    for comp in ("B4", "B4r"):
        for msl in (1, 2, 5):
            s.append((comp, True, dict(min_samples_leaf=msl)))
    return s

def conf_label(conf):
    return "_".join(f"{k}={v}" for k, v in conf.items())

def features(comp, D, window, cfg, cache):
    if comp == "B0b":
        return D["cycle"][:, None]
    if ("w", window) not in cache:
        io, ic, L = window_geom(window)
        col = np.asarray(D["qdlin"][:, ic], dtype=np.float64) - np.asarray(D["qdlin"][:, io], dtype=np.float64)
        cache[("w", window)] = col[D["row_ids"]]
    if comp in ("B3", "B3r"):
        return np.column_stack([cache[("w", window)], D["q0"].astype(np.float64)])
    if ("dq", window) not in cache:
        cache[("dq", window)] = build_dq(cfg, D, window)
    DQ = cache[("dq", window)]
    sub = mod.sub_idx(DQ.shape[1])
    return np.column_stack([DQ[:, sub].astype(np.float64), D["q0"].astype(np.float64)])

def fit_one(comp, conf, F, D, m, cfg):
    resid = comp in ("B3r", "B4r")
    tr, va = m["train_rows"], m["val_rows"]
    nom = cfg["NOMINAL_AH"]
    target = D["y"][tr] - (100.0 * D["q0"][tr].astype(np.float64) / nom if resid else 0.0)
    if comp.startswith("B4"):
        est = RandomForestRegressor(n_estimators=cfg["TREES"], min_samples_leaf=conf["min_samples_leaf"],
                                    max_features=0.33, n_jobs=10, random_state=cfg["SEED"])
    else:
        est = HistGradientBoostingRegressor(max_depth=conf["max_depth"], learning_rate=conf["learning_rate"],
                                            max_iter=200, early_stopping=False, random_state=cfg["SEED"])
    t = time.perf_counter()
    est.fit(F[tr], target, sample_weight=m["train_w"])
    sec = time.perf_counter() - t
    pred = est.predict(F[va]) + (100.0 * D["q0"][va].astype(np.float64) / nom if resid else 0.0)
    return pred, sec

def run_comparators(ctx):
    cfg, D, probs = ctx["cfg"], ctx["D"], ctx["problems"]
    info = ctx["val_info"]
    members = {p["pid"]: make_member(D, p, cfg["SEED"], cfg) for p in probs}
    for p in probs:
        assert_no_leak(D, p, members[p["pid"]])
    fcache, rows = {}, []
    for comp, windowed, conf in comp_specs():
        for window in (cfg["WINDOWS"] if windowed else [None]):
            k = dict(module=MODULE_HASH, comp=comp, window=window, conf=conf, trees=cfg["TREES"], seed=cfg["SEED"],
                     outer_sets=[list(x) for x in cfg["OUTER_SETS"]], stride=cfg["TRAIN_STRIDE"], sk=sklearn.__version__)
            path = cfg["cache"] / ("comp_" + hashlib.sha256(key_str(k).encode()).hexdigest()[:20] + ".json")
            res = None
            if path.exists():
                try:
                    z = json.loads(path.read_text())
                    if z["key"] == key_str(k):
                        res = z["rows"]
                except Exception:
                    res = None
            name = f"{comp} {window} {conf_label(conf)}"
            if res is None:
                try:
                    t0 = time.perf_counter()
                    F = features(comp, D, window, cfg, fcache)
                    res = []
                    for p in probs:
                        vi = info[p["pid"]]
                        pred, sec = fit_one(comp, conf, F, D, members[p["pid"]], cfg)
                        res.append(dict(pid=p["pid"], oi=p["oi"], mae=macro(pred, vi["y"], vi["codes"])[0],
                                        mae_ns=macro(pred, vi["y"], vi["codes"], vi["keep"])[0], fit_sec=sec))
                    path.write_text(json.dumps(dict(key=key_str(k), rows=res)))
                    log_run("cmp:" + comp, name, time.perf_counter() - t0, False)
                except Exception as e:
                    record_error("comparator " + name, e)
                    continue
            else:
                log_run("cmp:" + comp, name, 0.0, True)
            r = pd.DataFrame(res)
            n_out = len(cfg["OUTER_SETS"])
            outer = [float(r[r.oi == o].mae.mean()) for o in range(n_out)]
            rows.append(dict(comparator=comp, window=window or "-", config=conf_label(conf), mae=float(r.mae.mean()),
                             mae_ns=float(r.mae_ns.mean()), outer=outer, se=se(outer), fit_sec=float(r.fit_sec.mean())))
    cdf = pd.DataFrame(rows)
    sel = {}
    for (comp, window), g in cdf.groupby(["comparator", "window"]):
        sel[(comp, window)] = g.loc[g["mae"].idxmin()]
    cdf["selected"] = [sel[(r.comparator, r.window)]["config"] == r.config for r in cdf.itertuples()]
    ctx["comp_df"] = cdf
    ctx["comp_sel"] = pd.DataFrame([dict(r) for r in sel.values()])

def comp_value(ctx, comp, window):
    s = ctx["comp_sel"]
    row = s[(s.comparator == comp) & (s.window.isin([window, "-"]))]
    return None if row.empty else row.iloc[0]

## 5. Decisions

In [8]:
PREF = ["M1", "M1p", "M0"]

def decide(ctx):
    cfg, df = ctx["cfg"], ctx["pilot_df"]
    nwin = len(cfg["WINDOWS"])
    plan = dict(module_hash=MODULE_HASH, cfg=cfg["name"], notes=[])
    if df.empty:
        raise RuntimeError("no pilot results to decide on")
    combos = {}   # (model, head, lr) -> window-averaged values
    for (model, head, lr), g in df.groupby(["model", "head", "lr"]):
        if len(g) != nwin:
            plan["notes"].append(f"{model}-{head} lr={lr} missing a window; excluded")
            continue
        combos[(model, head, lr)] = dict(
            ens=float(g.ens_mae.mean()), single=float(g.single_mae.mean()),
            outer=np.mean([np.array(x) for x in g.ens_outer], axis=0))
    fam = {}
    for (model, head, lr), v in combos.items():
        if model not in fam or v["ens"] < fam[model][1]["ens"]:
            fam[model] = ((model, head, lr), v)
    order = sorted(fam, key=lambda m: fam[m][1]["ens"])
    best, second = order[0], (order[1] if len(order) > 1 else None)
    within = second is not None and fam[second][1]["ens"] <= 1.03 * fam[best][1]["ens"]
    primary = min([best, second], key=PREF.index) if within else best
    pk, pv = fam[primary]
    plan["family_best"] = {m: dict(head=fam[m][0][1], lr=fam[m][0][2], ens_mae=fam[m][1]["ens"], single_mae=fam[m][1]["single"]) for m in fam}
    plan["primary_model"] = primary; plan["primary_head"] = pk[1]
    plan["primary_within_3pct_tiebreak_used"] = bool(within)
    # lr for Phase 5b
    lrs = sorted(l for (m, h, l) in combos if m == primary and h == pk[1])
    vals = {l: combos[(primary, pk[1], l)] for l in lrs}
    lo = min(vals, key=lambda l: vals[l]["ens"])
    if len(lrs) > 1:
        other = [l for l in lrs if l != lo][0]
        diff, s_lo = vals[other]["ens"] - vals[lo]["ens"], se(vals[lo]["outer"])
        within_se = (not np.isfinite(s_lo)) or diff <= s_lo
        lr5b = 2e-3 if (within_se and 2e-3 in lrs) else lo
        plan["lr_detail"] = dict(lower_lr=lo, diff=diff, se=s_lo, within_one_se=bool(within_se))
    else:
        lr5b = lo
    plan["lr_phase5b"] = lr5b
    # ensemble gain
    plan["ensemble_gain"] = {m: dict(single=fam[m][1]["single"], ens=fam[m][1]["ens"], gain_pp=fam[m][1]["single"] - fam[m][1]["ens"],
                                     gain_pct=100 * (fam[m][1]["single"] - fam[m][1]["ens"]) / fam[m][1]["single"]) for m in fam}
    # M1p vs M1
    if "M1" in fam and "M1p" in fam:
        d = fam["M1"][1]["outer"] - fam["M1p"][1]["outer"]
        plan["m1p_vs_m1"] = dict(mean_gain_pp=float(d.mean()), se=se(d), m1p_beats_m1_by_gt_1se=bool(np.isfinite(se(d)) and d.mean() > se(d)))
    # ratios to comparators, per window
    plan["ratios"] = {}
    for w in cfg["WINDOWS"]:
        row = df[(df.model == primary) & (df["head"] == pk[1]) & (df.lr == pk[2]) & (df.window == w)]
        pm = float(row.ens_mae.iloc[0]) if len(row) else float("nan")
        entry = dict(primary_ens_mae=pm)
        for comp in ("B4", "B4r"):
            c = comp_value(ctx, comp, w)
            if c is not None:
                entry[comp] = dict(config=c["config"], mae=float(c["mae"]), ratio=pm / float(c["mae"]), le_0_85=bool(pm / float(c["mae"]) <= 0.85))
        plan["ratios"][w] = entry
    # compute estimate for Phase 5b (assumptions stated)
    spm = float(df[(df.model == primary) & (df["head"] == pk[1])].sec_per_member.mean())
    sets = int(ctx["D"]["pdD"][["repeat", "outer_fold"]].drop_duplicates().shape[0])
    seeds5b, problems_per_set = 5, 5
    cs = ctx["comp_sel"]
    cmp_sec = float(cs.fit_sec.sum()) if len(cs) else float("nan")
    plan["compute_estimate"] = dict(
        sec_per_member_fit=spm, comparator_fit_sec_sum_of_selected=cmp_sec,
        assumptions=f"{sets} outer sets x {problems_per_set} fits (4 inner + 1 outer-train) x {seeds5b} seeds x {len(cfg['WINDOWS'])} windows; "
                    f"comparators: one fit of each selected config per fit; seeds5b is an assumption, not a spec value",
        deep_minutes=sets * problems_per_set * seeds5b * len(cfg["WINDOWS"]) * spm / 60,
        comparator_minutes=sets * problems_per_set * cmp_sec / 60)
    plan["notes"].append("selection uses mean ensemble MAE over all windows and outer sets; best lr per model")
    ctx["plan"] = plan
    (cfg["out"] / "compute_plan2.json").write_text(json.dumps(plan, indent=2, default=float))

## 6. Figures

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def make_figures(ctx):
    cfg, df, plan = ctx["cfg"], ctx["pilot_df"], ctx["plan"]
    wins = cfg["WINDOWS"]
    variants = [f"{m}-{h}" for m, h in VARIANTS]
    colors = ["#4c78a8", "#9ecae9", "#e45756", "#54a24b"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    wd = 0.8 / len(VARIANTS)
    for j, (m, h) in enumerate(VARIANTS):
        vals, errs = [], []
        for w in wins:
            g = df[(df.model == m) & (df["head"] == h) & (df.window == w)]
            if g.empty:
                vals.append(np.nan); errs.append(0); continue
            best = g.loc[g.ens_mae.idxmin()]
            vals.append(best.ens_mae); errs.append(0 if not np.isfinite(best.ens_se) else best.ens_se)
        ax.bar(np.arange(len(wins)) + (j - 1.5) * wd, vals, wd, yerr=errs, capsize=3, label=f"{m}-{h}", color=colors[j])
    for comp, ls in (("B4", "--"), ("B4r", ":")):
        for i, w in enumerate(wins):
            c = comp_value(ctx, comp, w)
            if c is not None:
                ax.hlines(c["mae"], i - 0.45, i + 0.45, colors="black", linestyles=ls, label=comp if i == 0 else None)
    ax.set_xticks(range(len(wins))); ax.set_xticklabels(wins)
    ax.set_ylabel("mean inner cell-macro MAE (pp), 2-seed ensemble"); ax.legend(fontsize=8)
    ax.set_title("Inner-fold MAE by model and window (error bars: SE over outer sets)")
    fig.tight_layout(); fig.savefig(cfg["fig"] / "mae_by_model_window.png", dpi=130); plt.close(fig)

    lr = plan["lr_phase5b"]
    fig, ax = plt.subplots(figsize=(7, 4))
    w = wins[0]
    for model in KINDS:
        head = "abs"
        run = ctx["runs"].get((w, model, head, lr))
        if run is None:
            continue
        idx = [i for i, (pid, s) in enumerate(ctx["meta"]) if ctx["problems"][pid]["oi"] == 0]
        ax.plot(run["curve"]["epochs"], run["curve"]["val_mae"][:, idx].mean(1), marker="o", label=f"{model}-abs")
    ax.set_xlabel("epoch"); ax.set_ylabel("inner validation MAE (pp)"); ax.legend()
    ax.set_title(f"Learning curves, outer set {cfg['OUTER_SETS'][0]}, {w}, lr={lr}")
    fig.tight_layout(); fig.savefig(cfg["fig"] / "learning_curves.png", dpi=130); plt.close(fig)

## 7. Report

In [10]:
def fmt_df(df, cols):
    return df[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}")

def build_report(ctx):
    cfg = ctx["cfg"]; sc_name = cfg["name"]
    L = ["PHASE 5A2 REPORT", f"config: {sc_name}   runtime: {(time.time() - T0) / 60:.1f} min   module hash: {MODULE_HASH}",
         f"GPU: {GPU_INFO}", f"RAM: {RAM_INFO}", ""]
    V = ctx.get("verif")
    if V:
        L.append("VERIFICATION")
        for r in V["dq"]:
            L.append(f"  2a {r['window']}: block-vs-single max diff {r['max_abs_diff']:.2e}, delta-Q[0] max {r['max_abs_first']}, width {r['width']}, finite {r['finite']}")
        L.append(f"  2b assert_allowed(['q0']) roles {V['allowed']}")
        for r in V["parity"]:
            L.append(f"  2c {r['kind']} {r['dtype'].split('.')[-1]}: param diff {r['max_param_diff']:.2e}, eval diff {r['max_eval_diff']:.2e}")
        for r in V["counts"]:
            L.append(f"  2d {r['kind']}: {r['params']} params (expected {r['expected']}), batched G=3 {r['batched3']}")
        for r in V["seed"]:
            L.append(f"  2d {r['kind']}: same-seed identical {r['identical']}, other seed differs {r['differs_other_seed']}")
        L.append("")
    if "pilot_df" in ctx:
        L.append("MAIN RESULTS (inner cell-macro MAE, pp; SE across outer sets; ns = excluding stitched cells)")
        d = ctx["pilot_df"].sort_values(["window", "model", "head", "lr"]).copy()
        d["per_outer_ens"] = d.ens_outer.apply(lambda x: "/".join(f"{v:.3f}" for v in x))
        L.append(fmt_df(d, ["window", "model", "head", "lr", "single_mae", "single_se", "ens_mae", "ens_se", "train_mae",
                            "single_ns", "ens_ns", "median_cell", "p90_cell", "per_outer_ens", "sec_per_member"]))
        L.append("")
    if "comp_df" in ctx:
        L.append("COMPARATORS (selected config per comparator and window; mean over problems)")
        s = ctx["comp_sel"].copy()
        s["per_outer"] = s.outer.apply(lambda x: "/".join(f"{v:.3f}" for v in x))
        L.append(fmt_df(s.sort_values(["comparator", "window"]), ["comparator", "window", "config", "mae", "se", "mae_ns", "per_outer", "fit_sec"]))
        L.append("")
    if "plan" in ctx:
        L.append("DECISIONS"); L.append(json.dumps(ctx["plan"], indent=1, default=float)); L.append("")
    sl = sorted([r for r in RUNS if r["scope"] == sc_name and not r["cached"]], key=lambda r: -r["seconds"])[:5]
    L.append("SLOWEST RUNS: " + "; ".join(f"{r['name']} {r['seconds']:.0f}s" for r in sl))
    L.append("")
    mine = [c for c in CHECKS if c["scope"] in ("setup", sc_name)]
    for st in ("PASS", "WARN", "FAIL"):
        sub = [c for c in mine if c["status"] == st]
        L.append(f"CHECKS {st} ({len(sub)})")
        for c in sub:
            L.append(f"  {c['name']} {c['detail'][:150]}")
    errs = [e for e in ERRORS if e["scope"] in ("setup", sc_name)]
    L.append(f"ERRORS ({len(errs)})")
    for e in errs:
        L.append(f"  {e['stage']}: {e['error']}")
    L.append("ARTIFACTS")
    for base in (cfg["out"], SRC / "soh_deep_v2.py"):
        files = [base] if base.is_file() else sorted(p for p in base.rglob("*") if p.is_file() and "cache" not in p.parts and (sc_name == "smoke" or "smoke" not in p.parts))
        for p in files:
            L.append(f"  {p.relative_to(ROOT)}  {p.stat().st_size / 1024:.1f} KB")
    return "\n".join(L)

def write_reports(ctx):
    cfg = ctx["cfg"]
    cdf = ctx.get("comp_df")
    if cdf is not None and "pilot_df" in ctx:
        a = cdf.assign(kind="comparator").drop(columns=["outer"]).rename(columns={"comparator": "model"})
        b = ctx["pilot_df"].assign(kind="deep", config=lambda x: "lr=" + x.lr.astype(str) + " head=" + x["head"],
                                   mae=lambda x: x.ens_mae, mae_ns=lambda x: x.ens_ns, fit_sec=lambda x: x.sec_per_member)[
            ["model", "window", "config", "mae", "mae_ns", "fit_sec", "kind"]]
        pd.concat([a, b], ignore_index=True).to_csv(cfg["out"] / "pilot2_log.csv", index=False)
    txt = build_report(ctx)
    (cfg["out"] / "phase5a2_report.txt").write_text(txt, encoding="utf-8")
    (cfg["out"] / "verification_report.json").write_text(json.dumps(dict(
        module_hash=MODULE_HASH, config=cfg["name"], verification=ctx.get("verif"),
        checks=[c for c in CHECKS if c["scope"] in ("setup", cfg["name"])],
        errors=[{k: v for k, v in e.items() if k != "trace"} for e in ERRORS if e["scope"] in ("setup", cfg["name"])]),
        indent=1, default=str))
    return txt

def run_pipeline(cfg):
    global SCOPE
    SCOPE = cfg["name"]
    for d in (cfg["out"], cfg["fig"], cfg["cache"]):
        d.mkdir(parents=True, exist_ok=True)
    ctx = dict(cfg=cfg)
    def stage(name, fn):
        t = time.time()
        try:
            fn(ctx)
        except Exception as e:
            record_error(f"{cfg['name']}:{name}", e)
        print(f"-- stage {name} done in {time.time() - t:.0f}s")
    stage("load", lambda c: c.update(D=load_data()))
    if "D" not in ctx:
        return ctx
    stage("problems", lambda c: c.update(problems=build_problems(cfg, c["D"])))
    if "problems" not in ctx:
        return ctx
    stage("verify", verify)
    stage("pilot", run_pilot)
    stage("pilot_table", pilot_table)
    stage("comparators", run_comparators)
    stage("decisions", decide)           # writes compute_plan2.json before any figure
    stage("figures", make_figures)
    stage("report", write_reports)
    return ctx

## Run: smoke first, then full

In [11]:
SMOKE = run_pipeline(SMOKE_CFG)
smoke_errs = [e for e in ERRORS if e["scope"] == "smoke"]
smoke_fail = [c for c in CHECKS if c["scope"] == "smoke" and c["status"] == "FAIL"]
if smoke_errs or smoke_fail:
    raise RuntimeError(f"Smoke run not clean: {len(smoke_errs)} errors, {len(smoke_fail)} failed checks; see {OUT / 'smoke/phase5a2_report.txt'}")
print("SMOKE PASSED")

[PASS] labels rows == 104971 104971
[PASS] qdlin shape (104971, 1000) float32 (104971, 1000) float32
[PASS] cycles row_id == row position == qdlin row 
[PASS] labels row_id unique and within qdlin 
[PASS] qref shape (129, 1000) (129, 1000)
[PASS] qref_cell_ids row == position 
[PASS] protocol_D roles == {train, test} {'train', 'test'}
[PASS] protocol_D_inner has exactly two role values incl. 'train' {'val', 'train'}
[PASS] development cells: 84 (41 b1, 43 b2) 84 (41 b1, 43 b2)
[PASS] no sensitivity or batch-3 cell among development cells []
[PASS] every development cell has a qref row []
[PASS] development rows == 55963 55963
[PASS] finite q0 on development rows 
[PASS] finite soh_nom on development rows 
[PASS] finite cycle_number on development rows 
-- stage load done in 0s
[PASS] outer set (0, 0): 13-17 outer-test cells 14 (train 70)
[PASS] outer set (0, 0): train/test disjoint 
[PASS] outer set (0, 0): 4 inner folds [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
[PASS] outer

In [12]:
FULL = run_pipeline(FULL_CFG)

-- stage load done in 0s
[PASS] outer set (0, 0): 13-17 outer-test cells 14 (train 70)
[PASS] outer set (0, 0): train/test disjoint 
[PASS] outer set (0, 0): 4 inner folds [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
[PASS] outer set (0, 0): inner validation folds partition the outer-train cells 70 vs 70
[PASS] outer set (0, 1): 13-17 outer-test cells 14 (train 70)
[PASS] outer set (0, 1): train/test disjoint 
[PASS] outer set (0, 1): 4 inner folds [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
[PASS] outer set (0, 1): inner validation folds partition the outer-train cells 70 vs 70
[PASS] outer set (0, 2): 13-17 outer-test cells 14 (train 70)
[PASS] outer set (0, 2): train/test disjoint 
[PASS] outer set (0, 2): 4 inner folds [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
[PASS] outer set (0, 2): inner validation folds partition the outer-train cells 70 vs 70
-- stage problems done in 0s
[PASS] W300 geometry i_open=0 i_close=333 width=334
[PASS] W240 geometry i_open

In [13]:
SCOPE = "full"
cur, txt = gate_state()
check("gate sha256 unchanged and == gate_hash.txt (end)", cur == txt == GATE_START == GATE_SHA, cur[:16])
check("soh_deep_v2.py hash still matches (end)", sha256_file(MOD_PATH) == MODULE_HASH)
keep_awake(False)
print(write_reports(FULL))

[PASS] gate sha256 unchanged and == gate_hash.txt (end) 40df2fb82096e994
[PASS] soh_deep_v2.py hash still matches (end) 
PHASE 5A2 REPORT
config: full   runtime: 52.8 min   module hash: fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6.0 GB, cc 8.6, torch 2.14.1+cu130, cuda 13.0, TF32 matmul=False cudnn=False
RAM: 2.6 GB available; top: Code.exe 1.76GB, Code.exe 1.56GB, Code.exe 0.78GB, python.exe 0.64GB, explorer.exe 0.30GB, chrome.exe 0.29GB, MsMpEng.exe 0.27GB, chrome.exe 0.26GB

VERIFICATION
  2a W300: block-vs-single max diff 0.00e+00, delta-Q[0] max 0.0, width 334, finite True
  2a W240: block-vs-single max diff 0.00e+00, delta-Q[0] max 0.0, width 734, finite True
  2b assert_allowed(['q0']) roles ['primary']
  2c M0 float64: param diff 5.55e-17, eval diff 1.42e-14
  2c M1 float64: param diff 2.89e-10, eval diff 4.65e-10
  2c M1p float64: param diff 3.59e-11, eval diff 7.18e-11
  2c M0 float32: param diff 2.98e-08, eva